In [7]:
# =============================================================
# NOTEBOOK 01 – PREPROCESSING
# Môn: Học Máy MAT3533
# Thực hiện: Phan Thanh Lâm
# =============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import os
import json
import warnings
warnings.filterwarnings('ignore')

# ─── ĐƯỜNG DẪN FILE ─────────────────────────────
RAW_PATH = '../data/raw/IBM_HR_Attrition.csv'
CLEANED_PATH = '../data/processed/df_cleaned.csv'
SCALED_PATH = '../data/processed/df_scaled.csv'

print(f"✓ Đang làm việc tại: {os.getcwd()}")
print(f"✓ File dataset tồn tại: {os.path.exists(RAW_PATH)}")

# Nếu file không tồn tại, hiện danh sách file trong thư mục để kiểm tra
if not os.path.exists(RAW_PATH):
    print(f"\n⚠ Không tìm thấy file! Các file trong thư mục data/raw/:")
    if os.path.exists('data/raw'):
        print(os.listdir('data/raw'))
    else:
        print("Thư mục data/raw chưa được tạo hoặc đang trống")

# ═══════════════════════════════════════════════════════════
# BƯỚC 1 – ĐỌC DỮ LIỆU
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 1 – ĐỌC DỮ LIỆU")
print("=" * 60)

df_raw = pd.read_csv(RAW_PATH)
print(f"\n✓ Shape: {df_raw.shape[0]} mẫu × {df_raw.shape[1]} cột")

# Xem 5 dòng đầu
print("\n5 dòng đầu tiên:")
print(df_raw.head())

# ═══════════════════════════════════════════════════════════
# BƯỚC 2 – DROP CỘT VÔ NGHĨA (hằng số + ID)
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 2 – DROP CỘT HẰNG SỐ & ID")
print("=" * 60)

# Auto-detect cột hằng số (chỉ có 1 giá trị duy nhất)
constant_cols = [c for c in df_raw.columns if df_raw[c].nunique() == 1]
id_cols = ['EmployeeNumber']  # cột ID không có giá trị dự đoán

drop_cols = constant_cols + id_cols

print(f"\nCột hằng số (nunique=1):")
for c in constant_cols:
    print(f"  - {c}: giá trị = '{df_raw[c].iloc[0]}'")

print(f"\nCột ID:")
print(f"  - EmployeeNumber: {df_raw['EmployeeNumber'].nunique()} giá trị unique")

df = df_raw.drop(columns=drop_cols)
print(f"\n✓ Sau drop: {df.shape[0]} mẫu × {df.shape[1]} cột (đã xóa {len(drop_cols)} cột)")

# ═══════════════════════════════════════════════════════════
# BƯỚC 3 – KIỂM TRA MISSING VALUES
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 3 – KIỂM TRA MISSING VALUES")
print("=" * 60)

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'pct_%': missing_pct})
missing_cols = missing_df[missing_df['count'] > 0]

if len(missing_cols) == 0:
    print("\n✓ KHÔNG có missing values! Dataset rất sạch.")
else:
    print(f"\n⚠ Có {len(missing_cols)} cột missing:")
    print(missing_cols)

# ═══════════════════════════════════════════════════════════
# BƯỚC 4 – ENCODING CATEGORICAL FEATURES
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 4 – ENCODING CATEGORICAL FEATURES")
print("=" * 60)

df_encoded = df.copy()

# 4.1 Binary encoding (2 giá trị)
binary_cols = ['OverTime', 'Gender']
binary_map = {
    'OverTime': {'Yes': 1, 'No': 0},
    'Gender': {'Male': 1, 'Female': 0}
}
for col in binary_cols:
    df_encoded[col] = df[col].map(binary_map[col])
    print(f"✓ Binary   | {col:<15} → {binary_map[col]}")

# 4.2 Ordinal encoding (có thứ tự)
ordinal_map = {
    'BusinessTravel': {'Non-Travel': 0, 'Travel_Rarely': 1, 'Travel_Frequently': 2}
}
for col, mapping in ordinal_map.items():
    df_encoded[col] = df[col].map(mapping)
    print(f"✓ Ordinal  | {col:<15} → {mapping}")

# 4.3 One-hot encoding (nominal - không có thứ tự)
nominal_cols = ['Department', 'EducationField', 'JobRole', 'MaritalStatus']
df_encoded = pd.get_dummies(df_encoded, columns=nominal_cols, drop_first=False)
print(f"✓ One-hot  | {nominal_cols}")

# Chuyển bool thành int
bool_cols = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print(f"\n✓ Shape sau encoding: {df_encoded.shape}")
print(f"✓ Các cột hiện tại: {list(df_encoded.columns)[:10]}... (và {len(df_encoded.columns)-10} cột khác)")
print(f"✓ Target: MonthlyIncome (min={df_encoded['MonthlyIncome'].min():.0f}, max={df_encoded['MonthlyIncome'].max():.0f})")

# ═══════════════════════════════════════════════════════════
# BƯỚC 5 – XỬ LÝ OUTLIERS (phân tích, không xóa)
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 5 – PHÂN TÍCH OUTLIERS (IQR Method)")
print("=" * 60)

numeric_cols = df_encoded.select_dtypes(include=[np.number]).columns.tolist()
target = 'MonthlyIncome'
feature_num = [c for c in numeric_cols if c != target]

print(f"\n{'Feature':<25} {'Outliers':>8} {'Pct%':>6}   Quyết định")
print("-" * 50)

for col in feature_num:
    Q1 = df_encoded[col].quantile(0.25)
    Q3 = df_encoded[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df_encoded[col] < lower) | (df_encoded[col] > upper)).sum()
    pct = n_out / len(df_encoded) * 100
    
    if n_out > 0:
        decision = "GIỮ (tự nhiên)" if pct < 10 else "CAP"
        print(f"{col:<25} {n_out:>8} {pct:>5.1f}%   {decision}")

print("\n→ QUYẾT ĐỊNH: Giữ nguyên outliers")
print("  Lý do: Dữ liệu lương luôn có người lương cao (CEO, Director)")
print("  Xóa outlier sẽ làm mất thông tin thực tế")

# ═══════════════════════════════════════════════════════════
# BƯỚC 6 – CHUẨN HÓA (StandardScaler)
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 6 – STANDARDSCALER (chuẩn hóa về mean=0, std=1)")
print("=" * 60)

# Tách features và target
all_num = df_encoded.select_dtypes(include=[np.number]).columns.tolist()
scale_cols = [c for c in all_num if c != target]

scaler = StandardScaler()
df_scaled = df_encoded.copy()
df_scaled[scale_cols] = scaler.fit_transform(df_encoded[scale_cols])

print(f"✓ Đã chuẩn hóa {len(scale_cols)} features")
print(f"✓ Target (MonthlyIncome) giữ nguyên - không scale")

# Kiểm tra kết quả chuẩn hóa
print("\nKiểm tra (3 features đầu tiên):")
for col in scale_cols[:3]:
    print(f"  {col}: mean={df_scaled[col].mean():.4f}, std={df_scaled[col].std():.4f}")

# ═══════════════════════════════════════════════════════════
# BƯỚC 7 – LƯU FILE
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("BƯỚC 7 – LƯU FILE")
print("=" * 60)

# Lưu file cleaned (đã encode, chưa scale)
df_encoded.to_csv(CLEANED_PATH, index=False)
print(f"✓ Saved: {CLEANED_PATH}")

# Lưu file scaled (đã encode + scale)
df_scaled.to_csv(SCALED_PATH, index=False)
print(f"✓ Saved: {SCALED_PATH}")

# Lưu scaler params để sau còn dùng
scaler_params = {
    'feature_names': scale_cols,
    'mean_': scaler.mean_.tolist(),
    'scale_': scaler.scale_.tolist()
}
with open('../data/processed/scaler_params.json', 'w') as f:
    json.dump(scaler_params, f, indent=2)
print(f"✓ Saved: data/processed/scaler_params.json")

# ═══════════════════════════════════════════════════════════
# TÓM TẮT CUỐI CÙNG
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("📊 TÓM TẮT PREPROCESSING")
print("=" * 60)
print(f"""
  Dataset gốc         : {df_raw.shape}
  Sau khi xử lý       : {df_encoded.shape}
  Missing values      : 0 (không cần impute)
  Outliers            : Giữ nguyên (phân phối tự nhiên)
  Binary encoding     : OverTime, Gender
  Ordinal encoding    : BusinessTravel
  One-hot encoding    : Department, EducationField, JobRole, MaritalStatus
  StandardScaler      : {len(scale_cols)} features
  Target              : MonthlyIncome (không scale)
""")

print("=" * 60)
print("✅ notebook_01_preprocessing - HOÀN THÀNH!")

✓ Đang làm việc tại: c:\Users\Admin\OneDrive\Máy tính\ML_IBM_HR_Analytics\notebooks
✓ File dataset tồn tại: True

BƯỚC 1 – ĐỌC DỮ LIỆU

✓ Shape: 1470 mẫu × 35 cột

5 dòng đầu tiên:
   Age Attrition     BusinessTravel  DailyRate              Department  \
0   41       Yes      Travel_Rarely       1102                   Sales   
1   49        No  Travel_Frequently        279  Research & Development   
2   37       Yes      Travel_Rarely       1373  Research & Development   
3   33        No  Travel_Frequently       1392  Research & Development   
4   27        No      Travel_Rarely        591  Research & Development   

   DistanceFromHome  Education EducationField  EmployeeCount  EmployeeNumber  \
0                 1          2  Life Sciences              1               1   
1                 8          1  Life Sciences              1               2   
2                 2          2          Other              1               4   
3                 3          4  Life Sciences         